# 滑动窗口与累计窗口

学习目标：计算滑动与累计指标，按行数或时间定义窗口，核对端点、有效数量及结果与原记录的对应关系。

前置知识：时间索引、时间差、聚合、缺失值、分组与多级索引。

运行环境：Python 3.12、pandas 3.0；示例按 pandas 3.0.6 编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制测量，后续单元沿用已导入的 pd。时间示例明确使用 UTC。

## 1 计算最近三次平均值

rolling 创建滑动窗口，再用 mean 等方法对每个窗口计算。默认 center=False、closed="right" 时，整数窗口 3 表示当前行和前两行；结果放在当前行标签下。

下面观察五次温度测量，单位为摄氏度。默认需要三条有效观测才给出均值，因此开头不足三条的位置缺失。

In [1]:
import pandas as pd

temperature = pd.Series([18, 20, 22, 24, 26], index=["R1", "R2", "R3", "R4", "R5"], name="temperature_c")
moving = temperature.rolling(3).mean()
print(moving)  # R1、R2 缺失，R3 至 R5 依次为 20、22、24。
print(moving.index.equals(temperature.index), moving.dtype)  # True float64。
print((18 + 20 + 22) / 3)  # 20.0，手算核对 R3 的窗口。

R1     NaN
R2     NaN
R3    20.0
R4    22.0
R5    24.0
Name: temperature_c, dtype: float64
True float64
20.0


## 2 窗口成员与最少有效数量

### 2.1 先看窗口里有什么

窗口沿输入顺序移动，不会因为行标签看起来像时间就自动排序。迭代 rolling 对象可以查看各位置实际包含的记录；min_periods 控制能否产生聚合结果，不改变窗口成员。

下面继续使用 temperature，分别打印每个窗口的标签和值。

In [2]:
for window in temperature.rolling(3):
    print(window.index.tolist(), window.tolist())
# 窗口依次为 R1；R1-R2；R1-R3；R2-R4；R3-R5。
print(temperature.rolling(3, min_periods=1).mean().tolist())  # [18, 19, 20, 22, 24] 的浮点表示。

['R1'] [18]
['R1', 'R2'] [18, 20]
['R1', 'R2', 'R3'] [18, 20, 22]
['R2', 'R3', 'R4'] [20, 22, 24]
['R3', 'R4', 'R5'] [22, 24, 26]
[18.0, 19.0, 20.0, 22.0, 24.0]


### 2.2 缺失不算有效观测

对均值和求和，min_periods 表示至少需要多少个非缺失值。窗口里有三行不等于有三个有效数值；降低门槛会允许只根据部分记录计算，必须符合任务口径。

下面的 count 显示窗口内非缺失数量，与两种均值门槛放在一起比较。

In [3]:
readings = pd.Series([10.0, None, 30.0, 50.0], index=["A", "B", "C", "D"])
summary = pd.DataFrame({
    "valid_count": readings.rolling(3, min_periods=1).count(),
    "mean_at_least_2": readings.rolling(3, min_periods=2).mean(),
    "mean_all_3": readings.rolling(3, min_periods=3).mean(),
})
print(summary)
# 有效数量依次为 1、1、2、2；第二列为 NaN、NaN、20、40；第三列全为 NaN。
print(summary.shape)  # (4, 3)，每个结果仍对应原观测位置。

   valid_count  mean_at_least_2  mean_all_3
A          1.0              NaN         NaN
B          1.0              NaN         NaN
C          2.0             20.0         NaN
D          2.0             40.0         NaN
(4, 3)


## 3 行数窗口与时间窗口

整数窗口按记录位置取最近若干行；时间窗口如 "2min" 按时间范围取记录，行数可以变化。使用时间窗口需要日期时间索引，或在 DataFrame 的 on 参数指定时间列。时间顺序必须符合单调条件；本章统一按时间升序排列。

设当前时刻为 t，默认右闭的两分钟窗口为 (t−2 分钟, t]。下面记录间隔不规则，“最近两行”与“最近两分钟”会得到不同结果。

In [4]:
times = pd.to_datetime(["2026-01-01 00:00", "2026-01-01 00:01", "2026-01-01 00:04", "2026-01-01 00:05"], utc=True)
observations = pd.Series([10, 20, 40, 50], index=times, name="value")
comparison = pd.DataFrame({
    "last_2_rows": observations.rolling(2).mean(),
    "last_2_minutes": observations.rolling("2min").mean(),
})
print(comparison)
# 行数窗口：NaN、15、30、45；时间窗口：10、15、40、45。
# 00:04 的两分钟窗口只有该时刻一条观测，不含 00:01。
print(observations.index.dtype)  # 当前输入解析为 datetime64[us, UTC]，不假定总是 ns。

                           last_2_rows  last_2_minutes
2026-01-01 00:00:00+00:00          NaN            10.0
2026-01-01 00:01:00+00:00         15.0            15.0
2026-01-01 00:04:00+00:00         30.0            40.0
2026-01-01 00:05:00+00:00         45.0            45.0
datetime64[us, UTC]


### 3.1 两种默认 min_periods

整数窗口默认 min_periods 等于 window；时间长度窗口默认是 1。若比较两种窗口的统计结果，应同时检查门槛是否一致。

下面沿用 observations，让两分钟窗口也至少包含两条有效记录，并展示不规则时间输入需要明确整理顺序。

In [5]:
strict_time = observations.rolling("2min", min_periods=2).mean()
print(strict_time.tolist())  # [NaN, 15, NaN, 45]。
scrambled = observations.iloc[[0, 2, 1, 3]]
try:
    scrambled.rolling("2min").mean()
except ValueError:
    print("ValueError：时间索引不单调")
else:
    raise AssertionError("预期无序时间窗口失败")
print(scrambled.sort_index().rolling("2min").mean().equals(comparison["last_2_minutes"]))  # True。

[nan, 15.0, nan, 45.0]
ValueError：时间索引不单调
True


## 4 控制窗口端点

closed 决定左右边界是否计入。对于以 t 结尾、长度两分钟的非中心窗口，四种写法如下；圆括号不包含端点，方括号包含端点。

| closed | 中文名称／含义 | 时间范围 |
| --- | --- | --- |
| right | 左开右闭，默认 | (t−2 分钟, t] |
| left | 左闭右开 | [t−2 分钟, t) |
| both | 两端都包含 | [t−2 分钟, t] |
| neither | 两端都不包含 | (t−2 分钟, t) |

下面三条记录正好落在整分钟，在最后时刻逐一手算窗口和。

In [6]:
index = pd.date_range("2026-01-01", periods=3, freq="min", tz="UTC")
boundary = pd.Series([1, 2, 4], index=index)
for closed in ("right", "left", "both", "neither"):
    result = boundary.rolling("2min", closed=closed, min_periods=1).sum()
    print(closed, result.iloc[-1])
# 00:02：right 取 2+4=6；left 取 1+2=3；both 为 7；neither 只取 2。

right 6.0
left 3.0
both 7.0
neither 2.0


### 4.1 排除当前记录

实时计算常要求某时刻的指标只使用此前已知记录。closed="left" 可以排除右端当前时刻；但仍需确保时间字段表达的是实际可获得时间，而不只是事后补记的事件时间。

整数窗口也受 closed 影响，不能在改变端点后仍一概认为 window 就是实际成员数量。下面分别观察两种边界设置。

In [7]:
print(boundary.rolling("2min", closed="left", min_periods=1).sum().tolist())
# [NaN, 1, 3]：最后位置没有使用当前值 4。
row_values = pd.Series([1, 2, 3, 4])
print(row_values.rolling(2, closed="both", min_periods=1).sum().tolist())
# [1, 3, 6, 9]：后面的窗口包含三个位置，不能仍解释成恰好两条。

[nan, 1.0, 3.0]
[1.0, 3.0, 6.0, 9.0]


## 5 中心窗口与未来观测

center=True 将结果对应到窗口中心。对于按时间升序排列的记录，中心窗口可能使用标签时刻之后的观测，适合事后平滑展示，但不能直接当成当时已经可用的特征。

下面使用奇数长度 3，成员关系容易核对：中间位置对应前一条、当前条和后一条。

下图用五个不等间隔时刻对照窗口成员，横轴按记录顺序排列。居中三行窗口在中间时刻包含后一条记录，不能直接用于只能使用历史数据的预测任务。

![下图用五个不等间隔时刻对照窗口成员，横轴按记录顺序排列。居中三行窗口在中间时刻包含后一条记录，不能直接用于只能使用历史数据的预测任务。](image/17-window-members.png)

In [8]:
series = pd.Series([10, 20, 100, 40, 50], index=pd.date_range("2026-01-01", periods=5, freq="D", tz="UTC"))
trailing = series.rolling(3, center=False).mean()
centered = series.rolling(3, center=True).mean()
print(pd.DataFrame({"value": series, "trailing": trailing, "centered": centered}))
# 1 月 2 日中心均值约 43.333，已经使用 1 月 3 日的 100。
# 同一日尾随窗口仍缺失；两种结果不能只按标签日期视为相同可用性。
print(centered.iloc[1], series.iloc[:3].mean())  # 两者一致，明确用了哪三条记录。

                           value   trailing   centered
2026-01-01 00:00:00+00:00     10        NaN        NaN
2026-01-02 00:00:00+00:00     20        NaN  43.333333
2026-01-03 00:00:00+00:00    100  43.333333  53.333333
2026-01-04 00:00:00+00:00     40  53.333333  63.333333
2026-01-05 00:00:00+00:00     50  63.333333        NaN
43.333333333333336 43.333333333333336


## 6 时间列与多个统计量

DataFrame.rolling(on="time") 可按时间列建立窗口，结果仍按原行标签组织。选择数值列进行计算，并注明单位；时间列保留为时间信息，不作为数值求均值。

需要多个指标时可用 agg。标准差还要指定 ddof：默认 1 使用有效数量减 1 作分母，单个有效值无法形成这个分母；ddof=0 则使用有效数量本身。

In [9]:
table = pd.DataFrame({"time": times, "value": [10, 20, 40, 50]}, index=["R1", "R2", "R3", "R4"])
on_column = table.rolling("2min", on="time", min_periods=1).mean()
print(on_column)  # value 均值为 10、15、40、45，时间与 R1-R4 标签保留。
print(on_column.index.equals(table.index))  # True。
print(observations.rolling(2, min_periods=1).agg(["count", "mean"]))  # 每个窗口的有效数与均值。
print(observations.rolling(2, min_periods=1).std(ddof=1).iloc[0])  # NaN，只有一个有效值。
print(observations.rolling(2, min_periods=1).std(ddof=0).iloc[0])  # 0.0。

                        time  value
R1 2026-01-01 00:00:00+00:00   10.0
R2 2026-01-01 00:01:00+00:00   15.0
R3 2026-01-01 00:04:00+00:00   40.0
R4 2026-01-01 00:05:00+00:00   45.0
True
                           count  mean
2026-01-01 00:00:00+00:00    1.0  10.0
2026-01-01 00:01:00+00:00    2.0  15.0
2026-01-01 00:04:00+00:00    2.0  30.0
2026-01-01 00:05:00+00:00    2.0  45.0
nan
0.0


## 7 累计窗口

expanding 使用从开头到当前记录的全部前缀，窗口会不断扩大；rolling 只保留最近范围。expanding 默认 min_periods=1，均值等聚合按有效数值计算。

下面保留一条缺失记录，对照累计均值和最近两行均值。缺失位置仍可能得到累计统计值，因为前缀中已经存在有效观测；这不是填补原始测量。

In [10]:
daily = pd.Series([2.0, None, 6.0, 10.0], index=["day1", "day2", "day3", "day4"])
prefix = daily.expanding(min_periods=1)
print(prefix.agg(["count", "sum", "mean"]))
# 有效数量 1、1、2、3；合计 2、2、8、18；均值 2、2、4、6。
print(daily.rolling(2, min_periods=1).mean().tolist())  # [2, 2, 6, 8] 的浮点表示。
print(daily.expanding(min_periods=2).mean().tolist())  # [NaN, NaN, 4, 6]。
print(daily.isna().tolist())  # [False, True, False, False]，原缺失仍在。

      count   sum  mean
day1    1.0   2.0   2.0
day2    1.0   2.0   2.0
day3    2.0   8.0   4.0
day4    3.0  18.0   6.0
[2.0, 2.0, 6.0, 8.0]
[nan, nan, 4.0, 6.0]
[False, True, False, False]


## 8 选学：指数加权窗口

### 8.1 权重与 adjust

ewm 让较早观测的权重逐步衰减。下面直接给平滑系数 alpha，取值大于 0 且不超过 1；不提供 times 时，也可用 com、span 或 halflife 指定衰减，但只选其中一种。

adjust=True 把历史衰减权重归一化；adjust=False 使用递推，首项等于首个观测，随后为“(1−alpha) × 上一个结果 + alpha × 当前值”。下面输入无缺失，alpha=0.5，方便手算。

In [11]:
values = pd.Series([2.0, 4.0, 8.0])
adjusted = values.ewm(alpha=0.5, adjust=True).mean()
recursive = values.ewm(alpha=0.5, adjust=False).mean()
print(adjusted.tolist())  # 2、约 3.333、6；末项为 (0.25*2+0.5*4+8)/(0.25+0.5+1)。
print(recursive.tolist())  # [2.0, 3.0, 5.5]，依次与当前值各取一半。
print(values.ewm(alpha=0.5, min_periods=2).mean().tolist())  # 第一项缺失，后面同 adjusted。

[2.0, 3.3333333333333335, 6.0]
[2.0, 3.0, 5.5]
[nan, 3.3333333333333335, 6.0]


### 8.2 ignore_na 改变衰减距离

ignore_na=False 按原始位置间隔计算权重，True 按有效观测之间的相对位置计算；这项选择不等于把缺失值当成 0。

下面用 adjust=True、alpha=0.5，两个有效值之间隔一个缺失。未传 times 时衰减依据记录位置，不自动按真实时间间隔调整。

In [12]:
gapped = pd.Series([2.0, None, 8.0])
absolute = gapped.ewm(alpha=0.5, adjust=True, ignore_na=False).mean()
relative = gapped.ewm(alpha=0.5, adjust=True, ignore_na=True).mean()
print(absolute.tolist())  # [2, 2, 6.8]；末项 (0.25*2+8)/1.25。
print(relative.tolist())  # [2, 2, 6]；末项 (0.5*2+8)/1.5。
print(gapped.isna().tolist())  # 原始缺失没有被修改。

[2.0, 2.0, 6.8]
[2.0, 2.0, 6.0]
[False, True, False]


## 9 选学：分组窗口与结果对齐

先 groupby 再 rolling，会在各组内部建立窗口，组内按输入先后计算。输出通常包含“组键、原行标签”的 MultiIndex，不能直接把它当成与原表相同的索引。

下面原行标签全局唯一。移除组键层后按原索引 reindex，才能恢复交错输入顺序；若原行标签重复，应先建立唯一记录标识，不能套用这个简化处理。

In [13]:
devices = pd.DataFrame({"device": ["A", "B", "A", "B"], "value": [10, 100, 20, 200]},
                        index=["R1", "R2", "R3", "R4"])
grouped = devices.groupby("device", sort=False, observed=True, dropna=False)["value"].rolling(2, min_periods=1).mean()
print(grouped)  # A-R1 10、A-R3 15、B-R2 100、B-R4 150。
print(grouped.index.tolist())  # 顺序按组聚集，已经不是原来的交错顺序。
assert devices.index.is_unique
aligned = grouped.droplevel(0).reindex(devices.index)
result = devices.assign(moving_mean=aligned)
print(result)  # 按 R1-R4 恢复为 10、100、15、150，形状 (4, 3)。
print(result.index.equals(devices.index), result.shape)  # True (4, 3)。

device    
A       R1     10.0
        R3     15.0
B       R2    100.0
        R4    150.0
Name: value, dtype: float64
[('A', 'R1'), ('A', 'R3'), ('B', 'R2'), ('B', 'R4')]
   device  value  moving_mean
R1      A     10         10.0
R2      B    100        100.0
R3      A     20         15.0
R4      B    200        150.0
True (4, 3)


## 本章小结

（1）先确定窗口成员，再确定有效样本数门槛。行数窗口与时间窗口的范围和默认 min_periods 不同。

（2）closed 决定端点，center 决定结果对应位置；中心窗口可能使用未来数据。

（3）expanding 使用全部前缀；ewm 通过衰减权重汇总历史，adjust 与 ignore_na 改变具体口径。

（4）分组窗口不跨组，但结果索引可能新增层级。回填原表前检查唯一记录标识和标签顺序。

## 练习

（1）先预测以下两个均值结果，再运行。解释每个窗口里的总行数与有效值个数，以及默认 min_periods 为什么会影响输出。

In [14]:
sample = pd.Series([2.0, None, 6.0, 8.0])
print(sample.rolling(2).mean())
print(sample.rolling(2, min_periods=1).mean())
# 运行前按位置列出窗口成员，再核对缺失与均值。

0    NaN
1    NaN
2    NaN
3    7.0
dtype: float64
0    2.0
1    2.0
2    6.0
3    7.0
dtype: float64


（2）为不规则记录分别计算“最近两条”和“最近三分钟”的均值。要求至少两条有效记录，并解释两种结果为什么不能保证相同。

In [15]:
times = pd.to_datetime(["2026-01-01 00:00", "2026-01-01 00:01", "2026-01-01 00:05"], utc=True)
readings = pd.Series([10.0, 20.0, 50.0], index=times)
# 在此构造两种窗口，明确 min_periods 和 closed。
# 检查最后位置：两条窗口为 35，三分钟窗口因有效值不足而缺失。

（3）本来只做事后平滑，现要求每个时刻的指标仅使用此前已知记录，且不包含当前记录。选择合适的 center 与 closed，解释为什么中心窗口不再合适。

In [16]:
times = pd.date_range("2026-01-01", periods=4, freq="min", tz="UTC")
values = pd.Series([1, 2, 4, 8], index=times)
# 在此计算此前两分钟的合计，至少一个有效值；数据时间视为本题的可获得时间。
# 检查：结果为 NaN、1、3、6；最后窗口含 00:01 与 00:02，不含当前 00:03。

（4）为每台设备计算从首条到当前记录的累计均值，再放回原表。要求保留原交错顺序，解释选择 expanding 而不是固定长度 rolling 的理由。

In [17]:
records = pd.DataFrame({"device": ["A", "B", "A", "A"], "value": [2, 100, 4, 6]},
                        index=["R1", "R2", "R3", "R4"])
# 在此按组建立 expanding 窗口，再检查索引并对齐。
# 检查：原顺序累计均值为 2、100、3、4；R4 使用 A 组的全部三条记录。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Windowing operations](https://pandas.pydata.org/docs/user_guide/window.html) 的窗口迭代、Overview、Rolling window、Expanding window：有效值门槛、时间单调条件与分组窗口；[DataFrame.rolling](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) 的 window、min_periods、center、on、closed；[Rolling.count](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Rolling.count.html) 的非缺失计数；[Rolling.std](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Rolling.std.html) 的 ddof；[DataFrame.expanding](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.expanding.html) 的累计前缀与 min_periods；[DataFrame.ewm](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ewm.html) 的 alpha、衰减参数、adjust 公式、ignore_na 和 times 条件；[SeriesGroupBy.rolling](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.rolling.html) 的组内窗口与多级索引示例。实际可获得时间的约束是本例应用约定，未来观测是否被使用由上述窗口成员核对。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[window](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/window.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |